In [ ]:
import polars as pl

data = pl.read_parquet("../data/1d/1d.parquet").lazy()

Let's play with autogluon

In [ ]:
data.head().collect(engine="streaming")

In [ ]:
def backtest_top_k(
    data: pl.LazyFrame | pl.DataFrame = data,
    factor: str = "factor",
    k: int = 5,
    reverse: bool = False,
    fees: float = 0.0,
    plot=False,
) -> pl.LazyFrame:
    """
    topk factor backtest
    data: pl.LazyFrame with columns: open_time, return, factor
    factor: factor column name
    k: number of positions to long/short
    reverse: topk or bottomk, reverse=True means largest k
    fees: transaction fees per trade
    """

    cumsum = (
        data.lazy()
        .with_columns(
            pl.col("close")
            .sort_by("open_time")
            .pct_change()
            .shift(-1)
            .over("symbol")
            .alias("return")
        )
        .group_by("open_time")
        .agg(
            pl.col("return")
            .top_k_by(by=factor, k=k, reverse=reverse)
            .mean()
            .alias("strategy_return")
        )
        .sort("open_time")
        .with_columns(
            (pl.col("strategy_return") - fees).cum_sum().alias("cumulative_return")
        )
        .select(["open_time", "cumulative_return"])
    )

    if plot:
        cumsum.collect(engine="streaming").plot.line(
            x="open_time", y="cumulative_return"
        ).show()

    return cumsum

In [ ]:
backtest_top_k(
    data=data.with_columns(
        pl.col("close").pct_change().shift(-1).over("symbol").alias("factor")
    ),
    k=1,
    plot=True,
).collect(engine="streaming")